# destinepyauth Authentication Examples

[`destinepyauth`](https://github.com/SercoSPA/DestinE-Platform-AuthN) is a Python library for authenticating against Destination Earth Service Platform (DESP) services.

## How it works

The main entry point is `get_token(service)`, which returns an object containing an `access_token` and a `refresh:

```python
from destinepyauth import get_token

result = get_token("highway")   # prompts for username/password
token  = result.access_token   # use this as a Bearer token in HTTP requests
```

### Available services
| Service | Description |
|---|---|
| `cacheb` | CacheB data cache |
| `dea` | DEA service |
| `eden` | EDEN broker |
| `hda` | Harmonized Data Access |
| `highway` | Highway service |
| `insula` | Insula service |
| `polytope` | Polytope data access |
| `streamer` | Streaming service |

### Credential handling
When first called, `get_token()` prompts for your DESP username and password (masked input). You can also supply credentials via environment variables `DESPAUTH_USER` and `DESPAUTH_PASSWORD`.

The three examples below demonstrate authenticating against **HDA**, **EDEN**, and **Polytope**.

---
## Example 1 — HDA (Harmonized Data Access)

Authenticate against HDA and browse the STAC catalogue for CAMS Europe Air Quality Reanalyses.

In [ ]:
import requests
from destinepyauth import get_token

# Authenticate against HDA (includes automatic token exchange)
result = get_token("hda")
headers = {"Authorization": f"Bearer {result.access_token}"}

HDA_URL = "https://hda.data.destination-earth.eu/stac/v2"
COLLECTION_ID = "EO.AERIS.DAT.IAGOS"

# Fetch collection metadata
collection = requests.get(f"{HDA_URL}/collections/{COLLECTION_ID}", headers=headers).json()

print(f"Title:       {collection.get('title')}")
print(f"Description: {collection.get('description', '')[:200]}")
print(f"Extent:      {collection.get('extent', {}).get('temporal', {}).get('interval')}")

In [ ]:
# Search for the 5 most recent items in the collection
items = requests.post(
    f"{HDA_URL}/search",
    headers=headers,
    json={
        "collections": [COLLECTION_ID],
        "limit": 5,
    }
).json()

print(f"Matched items (total): {len(items)}")
for feat in items.get("features", []):
    props = feat.get("properties", {})
    print(f"  {feat['id']}  |  {props.get('datetime', props.get('start_datetime'))}")

---
## Example 2 — EDEN

Authenticate against EDEN and list available datasets, then search for recent products in a Climate DT dataset.

In [ ]:
import json
from destinepyauth import get_token

# Authenticate against EDEN
result = get_token("eden")
headers = {"Authorization": f"Bearer {result.access_token}"}

EDEN_URL = "https://broker.eden.destine.eu"

# List all available datasets
datasets = requests.get(f"{EDEN_URL}/api/v1/datasets").json()
print("Available datasets:")
for ds in datasets["features"]:
    print(f"  {ds['dataset_id']:60s}  {ds['metadata']['title']}")

In [ ]:
# Search for the 5 most recent ERA5 products
dataset_id = "EO:MEEO:DAT:REANALYSIS_ERA5_SINGLE_LEVELS:COG"

search_body = {
    "dataset_id": dataset_id,
    "itemsPerPage": 5,
    "startIndex": "0",
    "subDatasetId": "2m_temperature",
}
results = requests.post(
    f"{EDEN_URL}/api/v1/dataaccess/search",
    headers=headers,
    data=json.dumps(search_body),
).json()

print(f"Total products: {results['properties']['totalResults']}")
for feat in results["features"]:
    p = feat["properties"]
    print(f"  {feat['id']}  start={p['startdate']}  end={p['enddate']}")

---
## Example 3 — Polytope (Climate DT)

Authenticate against Polytope and retrieve Climate DT data with `earthkit-data`.

In [ ]:
import earthkit.data
from polytope.api import Client
from destinepyauth import get_token

# Authenticate — destinepyauth writes the refresh token to ~/.polytopeapirc automatically
get_token("polytope")

request = {
    "activity": "ScenarioMIP",
    "class": "d1",
    "dataset": "climate-dt",
    "date": "20200102",
    "experiment": "SSP3-7.0",
    "expver": "0001",
    "generation": "1",
    "levtype": "sfc",
    "model": "IFS-NEMO",
    "param": "134",          # Surface pressure
    "realization": "1",
    "resolution": "standard",
    "stream": "clte",
    "time": "0100",
    "type": "fc",
}

data = earthkit.data.from_source(
    "polytope",
    "destination-earth",
    request,
    address="polytope.lumi.apps.dte.destination-earth.eu",
    stream=False,
)
data.ls()